# ViT Experiments - Background Reliance
- Runs the ViT architecture, trained with and without background randomisation agumentation across two seeds

# Importing Libraries

In [2]:
import pathlib
import json
import torch
import numpy as np
import random
import pandas as pd

from cory_data_wrapper import *
from cory_utils import *
import tensorboard
import time

# Select Device

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device}")

Using cuda


# Create constants and create paths

In [4]:
# Make sure that the directory exists
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
SEEDS = [0, 339, 1]
VIT_ARCHITECTURE = ["vit_base_patch16_224"]


DATA_ROUTE = r"E:\AIML339\in9L_annotated_only"
BOUNDING_BOX_PATH = "LOC_train_solution.csv"
TEST_INSTANCES = pathlib.Path(r"E:\AIML339\backgrounds_challenge_data\bg_challenge")
RESULTS = pathlib.Path("./results")
TENSORBOARD_DIR = pathlib.Path("./runs")
RESULTS.mkdir(exist_ok=True)
TENSORBOARD_DIR.mkdir(exist_ok=True)

# Load the data

In [5]:
# Load data
bboxs = load_bbox_csv(BOUNDING_BOX_PATH)
base_train_dataset = IN9BoundingBoxDataset(DATA_ROUTE, bboxs, image_size=IMAGE_SIZE)
train_class_to_idx = base_train_dataset.class_to_idx

# print the instances and num of classes
print(f"Loaded {len(base_train_dataset)} images", "with" , len(base_train_dataset.classes), "classes")

Loaded 69120 images with 9 classes


# Split the data (Train, Validation)

In [6]:
train_subset, val_subset = stratified_train_val_split(base_train_dataset, val_fraction=0.1, seed=0)

# Create data loaders for all sets, for faster loading and shuffling and batches

# Create the validation loader
val_dataset_normalised = BackgroundRandomizationDataset(val_subset, augmentation_strength=0.0, seed=0, mean = VIT_MEAN, std = VIT_STD)
val_loader = make_loader(val_dataset_normalised, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# make loaders for the test set, one for each variant
test_variant_loaders = {
    variant: make_loader(
        fix_test_labels(
            ImageFolderEval(TEST_INSTANCES / variant / "val", image_size=IMAGE_SIZE, mean= VIT_MEAN, std= VIT_STD),train_class_to_idx
        ),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=0
    )
    for variant in ["original", "no_fg", "mixed_same", "mixed_rand"]
}
print("Test variants loaders created:", list(test_variant_loaders.keys()))

Test variants loaders created: ['original', 'no_fg', 'mixed_same', 'mixed_rand']


# Perform Learning Rate Search

- Perform a small learning rate search on Validation Accuracy
- Train for 3 epochs on 3 different learning rates.

In [20]:
# Create the dictionary for best Learning-Rate
BEST_LR = {
    "vit_base_patch16_224": 1e-5
}

best_lr_per_arch = {"vit_base_patch16_224": 1e-5}
# loop through all architectures and perform the Hyper Parameter search for each architecture
for arch_name in VIT_ARCHITECTURE:
    # Check to ensure that it only runs if LR not known,
    if BEST_LR[arch_name] is not None:
        best_lr_per_arch[arch_name] = BEST_LR[arch_name]
        continue

    print(f"\n{'-'*60}\n Hyper Parameter Search: {arch_name} \n{'-'*60}")
    #
    search_train_dataset = BackgroundRandomizationDataset(train_subset, augmentation_strength=0.0, seed=0, mean = VIT_MEAN, std = VIT_STD)
    search_train_loader = make_loader(search_train_dataset,batch_size=BATCH_SIZE, shuffle=True, num_workers=12, persistent_workers=True)

    best_lr, hyperparameter_trials = hyperparameter_search(
        lambda a = arch_name: ViTModel(a, num_classes=9, pretrained=True),
        search_train_loader, val_loader, device,
        learning_rates=(1e-5, 3e-5, 1e-4), search_epochs=3,
        log_dir=TENSORBOARD_DIR / "hpsearch" / arch_name
    )
    best_lr_per_arch[arch_name] = best_lr
    print(f"{arch_name} best LR: {best_lr:}")

print("\nBest LR per architecture:", best_lr_per_arch)


Best LR per architecture: {'vit_base_patch16_224': 1e-05}


In [8]:
# Architecture Train, Eval and Test Function

def run_vit_experiment(arch_name ="vit_base_patch16_224", augmented = False, seed= 0):
    """
    Run the ViT experiment for a specified architecture and seed.
    Stores the data in results

    @param arch_name: Name of the CNN architecture.
    @param augmented: Whether to use augmented or original data.
    @param seed: Seed for reproducibility.
    @return: A dictionary containing the results of the specified experiment.
    """
    data_type = "augmented" if augmented else "original"
    run_name = f"{arch_name}_{data_type}_{seed}"
    print(f"\n{'='*60}\n{run_name}\n{'='*60}")

    # If data is augmented, strength = 1 (swap) else 0 (do not swap background), swap the background with a random image's background
    train_dataset_for_run = BackgroundRandomizationDataset(
        train_subset, augmentation_strength=1.0 if augmented else 0.0, seed=seed, mean = VIT_MEAN, std = VIT_STD
    )
    train_loader = make_loader(train_dataset_for_run, batch_size=BATCH_SIZE, shuffle=True, num_workers=12, persistent_workers=True)

    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    # Define the model and move it to the device
    model = ViTModel(arch_name, num_classes=9, pretrained=True).to(device)

    start_time = time.time()
    # Train the model and store the results
    trained_model, history = train_model(
        model, train_loader, val_loader, device, epochs=EPOCHS, early_stopping_patience=3, model_name=run_name, lr=best_lr_per_arch[arch_name], warmup_fraction=0.15, log_dir=TENSORBOARD_DIR / run_name
    )
    training_time_in_secs = time.time() - start_time

    # Save the model weights
    weights_path = RESULTS / f"{run_name}_weights.pt"
    torch.save(trained_model.state_dict(), weights_path)


    variant_accuracies = evaluate_on_variants(trained_model, test_variant_loaders, device)
    # compute the reliance metrics
    reliance = compute_reliance_metrics(variant_accuracies, num_classes=9)
    print(f"Test variants: {variant_accuracies}")
    print(f"BG-Gap: {reliance['bg_gap']:.4f}, No-FG: {reliance['no_fg_accuracy']:.4f}, Original: {reliance['original_accuracy']:.4f}")
    # Store the results in a dictionary
    results = {
        "architecture" : arch_name,
        "data_type" : data_type,
        "seed" : seed,
        "variant_accuracies" : variant_accuracies,
        "reliance_metrics" : reliance,
        "final_val_accuracy" : history["val_accuracy"][-1],
        "epochs_run" : len(history["val_accuracy"]),
        "training_time_in_secs" : training_time_in_secs
    }
    # Save the dadta
    results_path = RESULTS / "vit_results.json"
    results_each_run = {}
    if results_path.exists():
        with open(results_path, "r") as f:
            for r in json.load(f):
                key = f"{r['architecture']}_{r['data_type']}_{r['seed']}"
                results_each_run[key] = r
    results_each_run[run_name] = results
    with open(results_path, "w") as f:
        json.dump(list(results_each_run.values()), f, indent=2)

    return results

# ViT Base (No Augmentation)

In [22]:
run_vit_experiment(arch_name= "vit_base_patch16_224",augmented=False, seed=0)


vit_base_patch16_224_original_0


C:\Users\minio\PycharmProjects\JupyterProject2\.venv\Lib\site-packages\torch\optim\lr_scheduler.py:1194: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


[vit_base_patch16_224_original_0] Epoch 1/10: train_loss=0.4876  train_acc=0.8349 val_loss=0.0384  val_acc=0.9907
[vit_base_patch16_224_original_0] Epoch 2/10: train_loss=0.0259  train_acc=0.9945 val_loss=0.0395  val_acc=0.9919
[vit_base_patch16_224_original_0] Epoch 3/10: train_loss=0.0118  train_acc=0.9971 val_loss=0.0523  val_acc=0.9916
[vit_base_patch16_224_original_0] Epoch 4/10: train_loss=0.0078  train_acc=0.9983 val_loss=0.0594  val_acc=0.9915
[vit_base_patch16_224_original_0] Early stopping at epoch 4
Test variants: {'original': 0.9940740740740741, 'no_fg': 0.7397530864197531, 'mixed_same': 0.9686419753086419, 'mixed_rand': 0.914074074074074}
BG-Gap: 0.0546, No-FG: 0.7398, Original: 0.9941


{'architecture': 'vit_base_patch16_224',
 'data_type': 'original',
 'seed': 0,
 'variant_accuracies': {'original': 0.9940740740740741,
  'no_fg': 0.7397530864197531,
  'mixed_same': 0.9686419753086419,
  'mixed_rand': 0.914074074074074},
 'reliance_metrics': {'chance_accuracy': 0.1111111111111111,
  'no_fg_accuracy': 0.7397530864197531,
  'bg_gap': 0.054567901234567895,
  'original_accuracy': 0.9940740740740741},
 'final_val_accuracy': 0.9914641203703703,
 'epochs_run': 4}

In [23]:
run_vit_experiment(arch_name= "vit_base_patch16_224",augmented=False, seed=339)


vit_base_patch16_224_original_339
[vit_base_patch16_224_original_339] Epoch 1/10: train_loss=0.5097  train_acc=0.8276 val_loss=0.0233  val_acc=0.9928
[vit_base_patch16_224_original_339] Epoch 2/10: train_loss=0.0259  train_acc=0.9938 val_loss=0.0415  val_acc=0.9910
[vit_base_patch16_224_original_339] Epoch 3/10: train_loss=0.0106  train_acc=0.9976 val_loss=0.0494  val_acc=0.9910
[vit_base_patch16_224_original_339] Early stopping at epoch 3
Test variants: {'original': 0.9923456790123457, 'no_fg': 0.7745679012345679, 'mixed_same': 0.965679012345679, 'mixed_rand': 0.9150617283950617}
BG-Gap: 0.0506, No-FG: 0.7746, Original: 0.9923


{'architecture': 'vit_base_patch16_224',
 'data_type': 'original',
 'seed': 339,
 'variant_accuracies': {'original': 0.9923456790123457,
  'no_fg': 0.7745679012345679,
  'mixed_same': 0.965679012345679,
  'mixed_rand': 0.9150617283950617},
 'reliance_metrics': {'chance_accuracy': 0.1111111111111111,
  'no_fg_accuracy': 0.7745679012345679,
  'bg_gap': 0.050617283950617264,
  'original_accuracy': 0.9923456790123457},
 'final_val_accuracy': 0.9910300925925926,
 'epochs_run': 3}

In [ ]:
run_vit_experiment(arch_name= "vit_base_patch16_224",augmented=False, seed=1)

# ViT Base (Augmentation)

In [24]:
run_vit_experiment(arch_name= "vit_base_patch16_224",augmented=True, seed=0)


vit_base_patch16_224_augmented_0
[vit_base_patch16_224_augmented_0] Epoch 1/10: train_loss=0.6787  train_acc=0.7749 val_loss=0.0658  val_acc=0.9829
[vit_base_patch16_224_augmented_0] Epoch 2/10: train_loss=0.1030  train_acc=0.9736 val_loss=0.0635  val_acc=0.9832
[vit_base_patch16_224_augmented_0] Epoch 3/10: train_loss=0.0595  train_acc=0.9851 val_loss=0.0818  val_acc=0.9805
[vit_base_patch16_224_augmented_0] Epoch 4/10: train_loss=0.0390  train_acc=0.9912 val_loss=0.0947  val_acc=0.9805
[vit_base_patch16_224_augmented_0] Early stopping at epoch 4
Test variants: {'original': 0.9903703703703703, 'no_fg': 0.5649382716049383, 'mixed_same': 0.9622222222222222, 'mixed_rand': 0.9590123456790124}
BG-Gap: 0.0032, No-FG: 0.5649, Original: 0.9904


{'architecture': 'vit_base_patch16_224',
 'data_type': 'augmented',
 'seed': 0,
 'variant_accuracies': {'original': 0.9903703703703703,
  'no_fg': 0.5649382716049383,
  'mixed_same': 0.9622222222222222,
  'mixed_rand': 0.9590123456790124},
 'reliance_metrics': {'chance_accuracy': 0.1111111111111111,
  'no_fg_accuracy': 0.5649382716049383,
  'bg_gap': 0.003209876543209811,
  'original_accuracy': 0.9903703703703703},
 'final_val_accuracy': 0.98046875,
 'epochs_run': 4}

In [25]:
run_vit_experiment(arch_name= "vit_base_patch16_224",augmented=True, seed=339)


vit_base_patch16_224_augmented_339


C:\Users\minio\PycharmProjects\JupyterProject2\.venv\Lib\site-packages\torch\optim\lr_scheduler.py:1194: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


[vit_base_patch16_224_augmented_339] Epoch 1/10: train_loss=0.6831  train_acc=0.7725 val_loss=0.0504  val_acc=0.9870
[vit_base_patch16_224_augmented_339] Epoch 2/10: train_loss=0.0997  train_acc=0.9749 val_loss=0.0589  val_acc=0.9861
[vit_base_patch16_224_augmented_339] Epoch 3/10: train_loss=0.0575  train_acc=0.9858 val_loss=0.0742  val_acc=0.9818
[vit_base_patch16_224_augmented_339] Early stopping at epoch 3
Test variants: {'original': 0.9874074074074074, 'no_fg': 0.6254320987654322, 'mixed_same': 0.9617283950617284, 'mixed_rand': 0.9471604938271605}
BG-Gap: 0.0146, No-FG: 0.6254, Original: 0.9874


{'architecture': 'vit_base_patch16_224',
 'data_type': 'augmented',
 'seed': 339,
 'variant_accuracies': {'original': 0.9874074074074074,
  'no_fg': 0.6254320987654322,
  'mixed_same': 0.9617283950617284,
  'mixed_rand': 0.9471604938271605},
 'reliance_metrics': {'chance_accuracy': 0.1111111111111111,
  'no_fg_accuracy': 0.6254320987654322,
  'bg_gap': 0.01456790123456786,
  'original_accuracy': 0.9874074074074074},
 'final_val_accuracy': 0.9817708333333334,
 'epochs_run': 3}

In [ ]:
run_vit_experiment(arch_name= "vit_base_patch16_224",augmented=True, seed=1)

# Short Results Summary

In [26]:
results_path = RESULTS / "vit_results.json"
with open(results_path, "r") as f:
    results = json.load(f)

rows = []
for r in results:
    row = {"architecture": r["architecture"], "data_type": r["data_type"], "seed": r["seed"], "final_val_accuracy": r["final_val_accuracy"], "epochs_run": r["epochs_run"]}
    row.update(r["reliance_metrics"])
    row["original_accuracy"] = r["variant_accuracies"].get("original")
    rows.append(row)

df = pd.DataFrame(rows)
summary = df.groupby(["architecture", "data_type"]).agg(
    mean_bg_gap =("bg_gap", "mean"), std_bg_gap =("bg_gap", "std"),
    mean_no_fg =("no_fg_accuracy", "mean"), std_no_fg =("no_fg_accuracy", "std"),
    mean_original =("original_accuracy", "mean"), std_original =("original_accuracy", "std"),
)
summary

mean_bg_gap  std_bg_gap  mean_no_fg  \
architecture         data_type                                        
vit_base_patch16_224 augmented     0.008889    0.008031    0.595185   
                     original      0.052593    0.002794    0.757160   

                                std_no_fg  mean_original  std_original  
architecture         data_type                                          
vit_base_patch16_224 augmented   0.042776       0.988889      0.002095  
                     original    0.024618       0.993210      0.001222

# Show TensorBoard

In [27]:
# %load_ext tensorboard
# %tensorboard --logdir ./runs


# Attention Rollout

In [13]:
from cory_attention_rollout import get_vit_attention_rollout
from PIL import Image
import pathlib
import torch

model = ViTModel("vit_base_patch16_224", num_classes=9, pretrained=False)
# Print out the parameter count
print(f"ViT-B/16 Parameter Count: {sum(p.numel() for p in model.parameters()):,}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESULTS = pathlib.Path("./results")

bboxs = load_bbox_csv(r"LOC_train_solution.csv")
base_data = IN9BoundingBoxDataset(r"E:\AIML339\in9L_annotated_only", bboxs, image_size=224)
eval_ds = BackgroundRandomizationDataset(base_data, augmentation_strength=0.0, seed=0, mean = VIT_MEAN, std = VIT_STD)

def perform_attn_rollout(augmented=False, img_index=339):
    if augmented:
        model.load_state_dict(torch.load(RESULTS / "vit_base_patch16_224_augmented_0_weights.pt"))
    else:
        model.load_state_dict(torch.load(RESULTS / "vit_base_patch16_224_original_0_weights.pt"))
    model.to(device)
    image, label = eval_ds[img_index]

    overlay, predicted = get_vit_attention_rollout(model, image, device = device)
    print(f"True Label: {label}, Predicted: {predicted}")
    Image.fromarray(overlay).save(f"attn_rollout_{'augmented' if augmented else 'original'}_{img_index}.png")

perform_attn_rollout(augmented=False, img_index=339)
perform_attn_rollout(augmented= True, img_index=339)

True Label: 0, Predicted: 0
True Label: 0, Predicted: 0


In [10]:
# import torch, random
# from cory_data_wrapper import grabcut_mask_from_bbox, build_mask_from_bbox
#
# def diagnose_augmentation(dataset, n_samples=500, seed=0, min_donor_background_fraction=0.3, use_grabcut=True):
#     py_rng = random.Random(seed)
#     n = len(dataset)
#     indices = py_rng.sample(range(len(dataset)), n)
#
#     n_swapped = 0
#     n_grabcut_success = 0
#
#     for idx in indices:
#         image, label, bbox = dataset[idx]
#         H, W = image.shape[1], image.shape[2]
#
#         if use_grabcut:
#             _, grabcut_succeeded = grabcut_mask_from_bbox(image, bbox)
#             if grabcut_succeeded:
#                 n_grabcut_success += 1
#
#         swapped_this_one = False
#         for _ in range(5):
#             bg_index = py_rng.randrange(len(dataset))
#             if bg_index == idx and len(dataset) > 1:
#                 continue
#             _, _, candidate_bbox = dataset[bg_index]
#             candidate_mask = build_mask_from_bbox(candidate_bbox, (H, W))
#             if candidate_mask.mean().item() >= min_donor_background_fraction:
#                 swapped_this_one = True
#                 break
#         if swapped_this_one:
#             n_swapped += 1
#
#     print(f"Sampled {len(dataset)} images:")
#     print(f"  Background swap actually applied: {n_swapped}/{n} ({n_swapped/n:.1%})")
#     print(f"  Silently unchanged (all 5 donors rejected): {n - n_swapped}/{n} ({(n-n_swapped)/n:.1%})")
#     if use_grabcut:
#         print(f"  GrabCut succeeded: {n_grabcut_success}/{n} ({n_grabcut_success/n:.1%})")
#         print(f"  GrabCut fell back to bbox: {n - n_grabcut_success}/{n} ({(n-n_grabcut_success)/n:.1%})")
#
# diagnose_augmentation(train_subset, n_samples=500, seed=0)

Sampled 62208 images:
  Background swap actually applied: 62180/62208 (100.0%)
  Silently unchanged (all 5 donors rejected): 28/62208 (0.0%)
  GrabCut succeeded: 52329/62208 (84.1%)
  GrabCut fell back to bbox: 9879/62208 (15.9%)
